# 从零实现 DiffPool：可微分簇分配与两层层次图分类

本 Notebook 只用 PyTorch 基础张量和 `nn.Module`，手写 dense GCN、assignment network、masked row softmax、$S^TX$、$S^TAS$、link-prediction auxiliary loss、entropy regularizer，以及两层 hierarchical DiffPool；不使用 PyG、DGL 或任何现成 GNN/池化层。

可执行合同覆盖：多图 padding、邻接对称性、assignment 行和、节点置换不变性、padding 不污染、退化空 cluster、跨图边拒绝、受控训练和带外发布信任。合成环/星图只用于验证实现，不等价于真实图分类 benchmark。

参考：[Hierarchical Graph Representation Learning with Differentiable Pooling, NeurIPS 2018](https://arxiv.org/abs/1806.08804)。其他层次池化设计可对照 [Graph U-Nets](https://arxiv.org/abs/1905.05178) 与采用 SortPooling 的 [DGCNN](https://arxiv.org/abs/1801.07829)。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import copy  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 5101  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_digest(payload) -> str:  # 定义本节可复用的核心函数。
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()  # 返回当前分支计算出的结果。

def state_digest51(state: dict[str, torch.Tensor]) -> str:  # 定义本节可复用的核心函数。
    h = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        value = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        h.update(key.encode()); h.update(str(value.dtype).encode())  # 执行当前语句以推进本节示例。
        h.update(str(tuple(value.shape)).encode()); h.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return h.hexdigest()  # 返回当前分支计算出的结果。

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert torch.initial_seed() == SEED  # 用受控断言验证关键不变量。
assert state_digest51({"x": torch.tensor([1], dtype=torch.long)}) != state_digest51({"x": torch.tensor([1.0])})  # 用受控断言验证关键不变量。
assert not any(name in globals() for name in ("torch_geometric", "dgl"))  # 用受控断言验证关键不变量。


## 1. 图级切分与输入合同

类别 0 是环，类别 1 是星形，节点数 5–8。节点特征为常数、归一化 degree、局部奇偶标记；degree 由当前图拓扑计算，不直接写入 label。每类 18 张图：前 12 张 train、随后 3 张 validation、最后 3 张 test，整张图只属于一个 split。

原始无向邻接必须有限、非负、对称且对角为 0。模型内部 pooling 后允许加权自环，但输入边不允许越界、重复或自环。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class GraphItem:  # 定义承载本节状态与行为的数据结构。
    graph_id: str  # 执行当前语句以推进本节示例。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    edge_pairs: tuple[tuple[int, int], ...]  # 执行当前语句以推进本节示例。
    label: int  # 执行当前语句以推进本节示例。
    split: str  # 执行当前语句以推进本节示例。

def make_graph51(label: int, index: int, split: str) -> GraphItem:  # 定义本节可复用的核心函数。
    if label not in (0, 1) or split not in {"train", "val", "test"}:  # 按当前条件选择后续控制路径。
        raise ValueError("label/split 非法")  # 遇到非法合同立即显式失败。
    n = 5 + index % 4  # 计算并保存当前步骤的中间状态。
    if label == 0:  # 按当前条件选择后续控制路径。
        pairs = {tuple(sorted((i, (i + 1) % n))) for i in range(n)}  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        pairs = {(0, i) for i in range(1, n)}  # 计算并保存当前步骤的中间状态。
    degree = torch.zeros(n)  # 计算并保存当前步骤的中间状态。
    for u, v in pairs:  # 遍历输入元素以累积或检查结果。
        degree[u] += 1; degree[v] += 1  # 计算并保存当前步骤的中间状态。
    x = torch.stack([torch.ones(n), degree / (n - 1), 0.1 * (torch.arange(n) % 2)], dim=-1)  # 计算并保存当前步骤的中间状态。
    return GraphItem(f"{split}-c{label}-{index:02d}", x.float(), tuple(sorted(pairs)), label, split)  # 返回当前分支计算出的结果。

def validate_graph_item51(graph: GraphItem) -> None:  # 定义本节可复用的核心函数。
    if graph.x.ndim != 2 or graph.x.shape[0] == 0 or not torch.isfinite(graph.x).all():  # 按当前条件选择后续控制路径。
        raise ValueError("节点特征必须是非空有限二维张量")  # 遇到非法合同立即显式失败。
    n, seen = graph.x.shape[0], set()  # 计算并保存当前步骤的中间状态。
    for u, v in graph.edge_pairs:  # 遍历输入元素以累积或检查结果。
        if not (0 <= u < n and 0 <= v < n) or u == v:  # 按当前条件选择后续控制路径。
            raise ValueError("边越界或含自环")  # 遇到非法合同立即显式失败。
        key = tuple(sorted((u, v)))  # 计算并保存当前步骤的中间状态。
        if key in seen: raise ValueError("无向边重复")  # 按当前条件选择后续控制路径。
        seen.add(key)  # 执行当前语句以推进本节示例。

graphs51 = []  # 计算并保存当前步骤的中间状态。
for label in (0, 1):  # 遍历输入元素以累积或检查结果。
    for index in range(18):  # 遍历输入元素以累积或检查结果。
        split = "train" if index < 12 else ("val" if index < 15 else "test")  # 计算并保存当前步骤的中间状态。
        graphs51.append(make_graph51(label, index, split))  # 执行当前语句以推进本节示例。
split_graphs51 = {s: [g for g in graphs51 if g.split == s] for s in ("train", "val", "test")}  # 计算并保存当前步骤的中间状态。
for graph in graphs51: validate_graph_item51(graph)  # 遍历输入元素以累积或检查结果。
assert tuple(len(split_graphs51[s]) for s in ("train", "val", "test")) == (24, 6, 6)  # 用受控断言验证关键不变量。
assert len({g.graph_id for g in graphs51}) == 36  # 用受控断言验证关键不变量。
assert set(g.label for g in split_graphs51["test"]) == {0, 1}  # 用受控断言验证关键不变量。
assert not ({g.graph_id for g in split_graphs51["train"]} & {g.graph_id for g in split_graphs51["test"]})  # 用受控断言验证关键不变量。


## 2. 多图 padded batch 与跨图边隔离

batch 张量为 `x:[B,Nmax,F]`、`adj:[B,Nmax,Nmax]`、`mask:[B,Nmax]`。padding 的 feature 和任一相关邻接必须为 0。`mask` 决定所有归一化、softmax、辅助损失和 readout 的有效范围。

若上游先构建全局稀疏边再 densify，必须检查每条边两端 `node_graph` 相同；否则跨图边会被静默写入错误样本。这里提供显式 ingestion oracle。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class DenseBatch:  # 定义承载本节状态与行为的数据结构。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    adj: torch.Tensor  # 执行当前语句以推进本节示例。
    mask: torch.Tensor  # 执行当前语句以推进本节示例。
    labels: torch.Tensor  # 执行当前语句以推进本节示例。
    graph_ids: tuple[str, ...]  # 执行当前语句以推进本节示例。
    snapshot_digest: str  # 执行当前语句以推进本节示例。

def validate_disjoint_edges51(edge_index: torch.Tensor, node_graph: torch.Tensor) -> None:  # 定义本节可复用的核心函数。
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_index.dtype != torch.long:  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 必须是 long[2,E]")  # 遇到非法合同立即显式失败。
    if node_graph.ndim != 1 or node_graph.dtype != torch.long:  # 按当前条件选择后续控制路径。
        raise ValueError("node_graph 必须是一维 long")  # 遇到非法合同立即显式失败。
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= node_graph.numel()):  # 按当前条件选择后续控制路径。
        raise ValueError("稀疏边端点越界")  # 遇到非法合同立即显式失败。
    if edge_index.numel() and not torch.equal(node_graph[edge_index[0]], node_graph[edge_index[1]]):  # 按当前条件选择后续控制路径。
        raise ValueError("检测到跨图边")  # 遇到非法合同立即显式失败。

def dense_snapshot_digest51(x, adj, mask, labels, graph_ids) -> str:  # 定义本节可复用的核心函数。
    tensors = {"x": x, "adj": adj, "mask": mask, "labels": labels}  # 计算并保存当前步骤的中间状态。
    return canonical_digest({"state": state_digest51(tensors), "graph_ids": list(graph_ids)})  # 返回当前分支计算出的结果。

def pad_graphs51(items: list[GraphItem], pad_to: int | None = None) -> DenseBatch:  # 定义本节可复用的核心函数。
    if not items: raise ValueError("batch 不能为空")  # 按当前条件选择后续控制路径。
    if len({g.graph_id for g in items}) != len(items): raise ValueError("graph_id 重复")  # 按当前条件选择后续控制路径。
    for graph in items: validate_graph_item51(graph)  # 遍历输入元素以累积或检查结果。
    feature_dim = items[0].x.shape[1]  # 计算并保存当前步骤的中间状态。
    if any(g.x.shape[1] != feature_dim for g in items): raise ValueError("特征维度不一致")  # 按当前条件选择后续控制路径。
    max_nodes = max(g.x.shape[0] for g in items)  # 计算并保存当前步骤的中间状态。
    pad_to = max_nodes if pad_to is None else pad_to  # 计算并保存当前步骤的中间状态。
    if pad_to < max_nodes: raise ValueError("pad_to 小于真实节点数")  # 按当前条件选择后续控制路径。
    x = torch.zeros(len(items), pad_to, feature_dim)  # 计算并保存当前步骤的中间状态。
    adj = torch.zeros(len(items), pad_to, pad_to)  # 计算并保存当前步骤的中间状态。
    mask = torch.zeros(len(items), pad_to, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for b, graph in enumerate(items):  # 遍历输入元素以累积或检查结果。
        n = graph.x.shape[0]; x[b, :n] = graph.x; mask[b, :n] = True  # 计算并保存当前步骤的中间状态。
        for u, v in graph.edge_pairs:  # 遍历输入元素以累积或检查结果。
            adj[b, u, v] = 1.0; adj[b, v, u] = 1.0  # 计算并保存当前步骤的中间状态。
    labels = torch.tensor([g.label for g in items], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    ids = tuple(g.graph_id for g in items)  # 计算并保存当前步骤的中间状态。
    digest = dense_snapshot_digest51(x, adj, mask, labels, ids)  # 计算并保存当前步骤的中间状态。
    return DenseBatch(x, adj, mask, labels, ids, digest)  # 返回当前分支计算出的结果。

train51 = pad_graphs51(split_graphs51["train"])  # 计算并保存当前步骤的中间状态。
val51 = pad_graphs51(split_graphs51["val"])  # 计算并保存当前步骤的中间状态。
test51 = pad_graphs51(split_graphs51["test"])  # 计算并保存当前步骤的中间状态。
assert train51.x.shape == (24, 8, 3) and train51.adj.shape == (24, 8, 8)  # 用受控断言验证关键不变量。
assert torch.equal(train51.adj, train51.adj.transpose(1, 2))  # 用受控断言验证关键不变量。
assert torch.count_nonzero(train51.x[~train51.mask]) == 0  # 用受控断言验证关键不变量。
assert torch.count_nonzero(train51.adj * (~train51.mask)[:, :, None]) == 0  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    validate_disjoint_edges51(torch.tensor([[0, 1], [1, 2]]), torch.tensor([0, 0, 1, 1]))  # 执行当前语句以推进本节示例。
    raise AssertionError("跨图边未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "跨图边" in str(exc)  # 用受控断言验证关键不变量。


## 3. 手写 dense GCN

对每张 padded 图，仅在有效节点加入自环，计算

$$\hat A=A+I_{valid},\qquad \tilde A=D^{-1/2}\hat A D^{-1/2},\qquad H'=\sigma(\tilde AHW).$$

mask 在归一化前后都应用；否则 padding 节点的 linear bias 或伪自环会污染后续 assignment。输入邻接允许 DiffPool 产生的非负权重和对角项，但始终要求对称、有限、padding 区域为零。对称合同使用绝对容差 `1e-5` 且 `rtol=0`（覆盖两次 float32 矩阵乘的舍入误差），避免大权重把明显绝对误差藏进相对容差；本发布版本还把单边权重上限冻结为 $10^9$。


In [ ]:
SYMMETRY_ATOL51 = 1e-5  # 计算并保存当前步骤的中间状态。
MAX_ADJ_WEIGHT51 = 1e9  # 计算并保存当前步骤的中间状态。

def validate_dense_tensors51(x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor) -> None:  # 定义本节可复用的核心函数。
    if x.ndim != 3 or adj.ndim != 3 or mask.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("x/adj/mask 维度非法")  # 遇到非法合同立即显式失败。
    b, n, _ = x.shape  # 计算并保存当前步骤的中间状态。
    if adj.shape != (b, n, n) or mask.shape != (b, n) or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("x/adj/mask shape 或 dtype 不匹配")  # 遇到非法合同立即显式失败。
    if (not torch.is_floating_point(x) or not torch.is_floating_point(adj) or x.dtype != adj.dtype  # 按当前条件选择后续控制路径。
            or x.device != adj.device or mask.device != x.device):  # 计算并保存当前步骤的中间状态。
        raise ValueError("x/adj/mask 必须同设备，且 x/adj 为同 dtype 浮点张量")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(x).all() or not torch.isfinite(adj).all() or bool((adj < 0).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("特征/邻接含非有限值或负权")  # 遇到非法合同立即显式失败。
    if bool((adj > MAX_ADJ_WEIGHT51).any()):  # 按当前条件选择后续控制路径。
        raise ValueError(f"邻接权重超过发布上限 {MAX_ADJ_WEIGHT51:g}")  # 遇到非法合同立即显式失败。
    if not torch.allclose(adj, adj.transpose(1, 2), atol=SYMMETRY_ATOL51, rtol=0.0):  # 按当前条件选择后续控制路径。
        raise ValueError("邻接矩阵必须对称")  # 遇到非法合同立即显式失败。
    pair_mask = mask[:, :, None] & mask[:, None, :]  # 计算并保存当前步骤的中间状态。
    if torch.count_nonzero(x.masked_fill(mask[:, :, None], 0.0)):  # 按当前条件选择后续控制路径。
        raise ValueError("padding 特征必须为零")  # 遇到非法合同立即显式失败。
    if torch.count_nonzero(adj.masked_fill(pair_mask, 0.0)):  # 按当前条件选择后续控制路径。
        raise ValueError("padding 邻接必须为零")  # 遇到非法合同立即显式失败。
    if bool((mask.sum(1) == 0).any()): raise ValueError("batch 含空图")  # 按当前条件选择后续控制路径。

def normalized_adjacency51(adj: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
    dummy_x = torch.zeros(adj.shape[0], adj.shape[1], 1, device=adj.device, dtype=adj.dtype)  # 计算并保存当前步骤的中间状态。
    validate_dense_tensors51(dummy_x, adj, mask)  # 执行当前语句以推进本节示例。
    eye = torch.diag_embed(mask.to(adj.dtype))  # 计算并保存当前步骤的中间状态。
    pair_mask = mask[:, :, None] & mask[:, None, :]  # 计算并保存当前步骤的中间状态。
    with_self = (adj + eye) * pair_mask  # 计算并保存当前步骤的中间状态。
    degree = with_self.sum(-1)  # 计算并保存当前步骤的中间状态。
    inv_sqrt = torch.where(mask, degree.clamp_min(1e-12).rsqrt(), torch.zeros_like(degree))  # 计算并保存当前步骤的中间状态。
    return with_self * inv_sqrt[:, :, None] * inv_sqrt[:, None, :]  # 返回当前分支计算出的结果。

class DenseGCN(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim: int, output_dim: int, activate: bool = True):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if input_dim <= 0 or output_dim <= 0: raise ValueError("GCN 维度必须为正")  # 按当前条件选择后续控制路径。
        self.linear = nn.Linear(input_dim, output_dim)  # 计算并保存当前步骤的中间状态。
        self.activate = activate  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        validate_dense_tensors51(x, adj, mask)  # 执行当前语句以推进本节示例。
        support = self.linear(x)  # 计算并保存当前步骤的中间状态。
        out = torch.bmm(normalized_adjacency51(adj, mask), support)  # 计算并保存当前步骤的中间状态。
        if self.activate: out = F.relu(out)  # 按当前条件选择后续控制路径。
        if not torch.isfinite(out).all(): raise ValueError("GCN 输出含非有限值")  # 按当前条件选择后续控制路径。
        return out * mask[:, :, None]  # 返回当前分支计算出的结果。

gcn_probe51 = DenseGCN(3, 5)(train51.x[:2], train51.adj[:2], train51.mask[:2])  # 计算并保存当前步骤的中间状态。
assert gcn_probe51.shape == (2, 8, 5)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(gcn_probe51[~train51.mask[:2]]) == 0  # 用受控断言验证关键不变量。

bad_adj51 = train51.adj[:1].clone(); bad_adj51[0, 0, 2] = 0.5  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    DenseGCN(3, 4)(train51.x[:1], bad_adj51, train51.mask[:1])  # 执行当前语句以推进本节示例。
    raise AssertionError("非对称邻接未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "对称" in str(exc)  # 用受控断言验证关键不变量。

large_asym51 = torch.tensor([[[0.0, 100000000.0], [100000512.0, 0.0]]])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_dense_tensors51(torch.ones(1, 2, 1), large_asym51, torch.tensor([[True, True]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("大权重下绝对差 512 的非对称邻接被默认 rtol 放过")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "对称" in str(exc)  # 用受控断言验证关键不变量。
assert not torch.allclose(large_asym51, large_asym51.transpose(1, 2), atol=SYMMETRY_ATOL51, rtol=0.0)  # 用受控断言验证关键不变量。


## 4. Assignment、池化公式与辅助损失

assignment network 输出 `logits:[B,N,K]`，只对有效节点行做 softmax：有效行和为 1，padding 行严格为 0。随后

$$X' = S^T Z,\qquad A'=S^TAS.$$

link 辅助项逼近邻接：$\|A-SS^T\|_F^2$，只统计有效节点对；entropy 项 $-\sum_k S_{ik}\log S_{ik}$ 抑制完全模糊的分配。link 在平方前按每张图的最大绝对误差缩放到安全范围，求均值后再恢复量纲；配合邻接权重上限与终端有限性检查，避免有限的大权重在 float32 平方时静默变成 `inf`。二者是正则项，不应混入 test 标签。

若某 cluster 的总 assignment mass 为 0，它被标记为无效，输出行/列清零，不能除零或产生 NaN。


In [ ]:
def masked_assignment_softmax51(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
    if logits.ndim != 3 or mask.shape != logits.shape[:2] or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("assignment logits/mask 合同不匹配")  # 遇到非法合同立即显式失败。
    if logits.shape[-1] <= 0 or not torch.isfinite(logits).all():  # 按当前条件选择后续控制路径。
        raise ValueError("cluster 数或 logits 数值非法")  # 遇到非法合同立即显式失败。
    out = torch.zeros_like(logits)  # 计算并保存当前步骤的中间状态。
    out[mask] = torch.softmax(logits[mask], dim=-1)  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。

def diffpool_tensors51(z: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor,  # 定义本节可复用的核心函数。
                       assignment: torch.Tensor, eps: float = 1e-8):  # 计算并保存当前步骤的中间状态。
    validate_dense_tensors51(z, adj, mask)  # 执行当前语句以推进本节示例。
    if (not isinstance(eps, (int, float)) or isinstance(eps, bool)  # 按当前条件选择后续控制路径。
            or not np.isfinite(float(eps)) or float(eps) <= 0):  # 计算并保存当前步骤的中间状态。
        raise ValueError("eps 必须是有限正数")  # 遇到非法合同立即显式失败。
    if assignment.shape[:2] != mask.shape or assignment.ndim != 3 or not torch.isfinite(assignment).all():  # 按当前条件选择后续控制路径。
        raise ValueError("assignment shape/数值非法")  # 遇到非法合同立即显式失败。
    if (not torch.is_floating_point(assignment) or assignment.dtype != z.dtype  # 按当前条件选择后续控制路径。
            or assignment.device != z.device):  # 计算并保存当前步骤的中间状态。
        raise ValueError("assignment dtype/device 必须与节点表示一致")  # 遇到非法合同立即显式失败。
    if bool((assignment < 0).any()): raise ValueError("assignment 不能为负")  # 按当前条件选择后续控制路径。
    row_sum = assignment.sum(-1)  # 计算并保存当前步骤的中间状态。
    if not torch.allclose(row_sum[mask], torch.ones_like(row_sum[mask]), atol=1e-5):  # 按当前条件选择后续控制路径。
        raise ValueError("有效 assignment 行和必须为 1")  # 遇到非法合同立即显式失败。
    if torch.count_nonzero(assignment[~mask]): raise ValueError("padding assignment 必须为零")  # 按当前条件选择后续控制路径。
    pair_mask = mask[:, :, None] & mask[:, None, :]  # 计算并保存当前步骤的中间状态。
    z_clean = z * mask[:, :, None]  # 计算并保存当前步骤的中间状态。
    adj_clean = adj * pair_mask  # 计算并保存当前步骤的中间状态。
    pooled_x = torch.bmm(assignment.transpose(1, 2), z_clean)  # 计算并保存当前步骤的中间状态。
    pooled_adj = torch.bmm(torch.bmm(assignment.transpose(1, 2), adj_clean), assignment)  # 计算并保存当前步骤的中间状态。
    cluster_mass = assignment.sum(1)  # 计算并保存当前步骤的中间状态。
    cluster_mask = cluster_mass > eps  # 计算并保存当前步骤的中间状态。
    cluster_pair = cluster_mask[:, :, None] & cluster_mask[:, None, :]  # 计算并保存当前步骤的中间状态。
    pooled_x = pooled_x * cluster_mask[:, :, None]  # 计算并保存当前步骤的中间状态。
    pooled_adj = pooled_adj * cluster_pair  # 计算并保存当前步骤的中间状态。
    if not torch.isfinite(pooled_x).all() or not torch.isfinite(pooled_adj).all():  # 按当前条件选择后续控制路径。
        raise ValueError("DiffPool pooled tensor 含非有限值")  # 遇到非法合同立即显式失败。
    reconstructed = torch.bmm(assignment, assignment.transpose(1, 2))  # 计算并保存当前步骤的中间状态。
    error = adj_clean - reconstructed  # 计算并保存当前步骤的中间状态。
    # 每图先缩放到绝对值不超过约 1 再平方，最后恢复量纲；结合 1e9 输入上限可避免 float32 overflow。
    error_scale = error.detach().abs().amax((1, 2), keepdim=True).clamp_min(1.0)  # 计算并保存当前步骤的中间状态。
    scaled_squared = (error / error_scale).square() * pair_mask  # 计算并保存当前步骤的中间状态。
    link_per_graph = (scaled_squared.sum((1, 2)) / pair_mask.sum((1, 2)).clamp_min(1)) * error_scale.flatten().square()  # 计算并保存当前步骤的中间状态。
    entropy_node = -(assignment.clamp_min(1e-12).log() * assignment).sum(-1)  # 计算并保存当前步骤的中间状态。
    entropy_per_graph = (entropy_node * mask).sum(1) / mask.sum(1).clamp_min(1)  # 计算并保存当前步骤的中间状态。
    link_loss = link_per_graph.mean()  # 计算并保存当前步骤的中间状态。
    entropy_loss = entropy_per_graph.mean()  # 计算并保存当前步骤的中间状态。
    if not torch.isfinite(link_loss) or not torch.isfinite(entropy_loss):  # 按当前条件选择后续控制路径。
        raise ValueError("DiffPool 辅助损失含非有限值")  # 遇到非法合同立即显式失败。
    return pooled_x, pooled_adj, cluster_mask, link_loss, entropy_loss  # 返回当前分支计算出的结果。

logits_probe51 = torch.tensor([[[2.0, 0.0], [0.0, 2.0], [99.0, -99.0]]])  # 计算并保存当前步骤的中间状态。
mask_probe51 = torch.tensor([[True, True, False]])  # 计算并保存当前步骤的中间状态。
assign_probe51 = masked_assignment_softmax51(logits_probe51, mask_probe51)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(assign_probe51[0, :2].sum(-1), torch.ones(2))  # 用受控断言验证关键不变量。
assert torch.equal(assign_probe51[0, 2], torch.zeros(2))  # 用受控断言验证关键不变量。

z_deg51 = torch.tensor([[[1.0], [2.0], [0.0]]])  # 计算并保存当前步骤的中间状态。
adj_deg51 = torch.tensor([[[0., 1., 0.], [1., 0., 0.], [0., 0., 0.]]])  # 计算并保存当前步骤的中间状态。
mask_deg51 = torch.tensor([[True, True, False]])  # 计算并保存当前步骤的中间状态。
assign_deg51 = torch.tensor([[[1., 0.], [1., 0.], [0., 0.]]])  # 计算并保存当前步骤的中间状态。
px_deg51, pa_deg51, pm_deg51, link_deg51, ent_deg51 = diffpool_tensors51(  # 计算并保存当前步骤的中间状态。
    z_deg51, adj_deg51, mask_deg51, assign_deg51)  # 执行当前语句以推进本节示例。
assert pm_deg51.tolist() == [[True, False]]  # 用受控断言验证关键不变量。
assert torch.allclose(px_deg51[0, 0], torch.tensor([3.0]))  # 用受控断言验证关键不变量。
assert torch.allclose(pa_deg51[0, 0, 0], torch.tensor(2.0))  # 用受控断言验证关键不变量。
assert torch.equal(px_deg51[0, 1], torch.zeros(1))  # 用受控断言验证关键不变量。
assert torch.equal(pa_deg51[0, 1], torch.zeros(2)) and torch.isfinite(pa_deg51).all()  # 用受控断言验证关键不变量。
assert torch.allclose(link_deg51, torch.tensor(0.5)) and float(ent_deg51) == 0.0  # 用受控断言验证关键不变量。

extreme_adj51 = torch.tensor([[[0.0, torch.finfo(torch.float32).max / 2],  # 计算并保存当前步骤的中间状态。
                               [torch.finfo(torch.float32).max / 2, 0.0]]])  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    diffpool_tensors51(torch.ones(1, 2, 1), extreme_adj51, torch.tensor([[True, True]]),  # 执行当前语句以推进本节示例。
                       torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("可能令平方 link loss 溢出的极值邻接未 fail-closed")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "上限" in str(exc)  # 用受控断言验证关键不变量。
safe_large51 = torch.tensor([[[0.0, 1e8], [1e8, 0.0]]])  # 计算并保存当前步骤的中间状态。
safe_result51 = diffpool_tensors51(torch.ones(1, 2, 1), safe_large51, torch.tensor([[True, True]]),  # 计算并保存当前步骤的中间状态。
                                    torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]))  # 执行当前语句以推进本节示例。
assert torch.isfinite(safe_result51[1]).all() and torch.isfinite(safe_result51[3])  # 用受控断言验证关键不变量。


## 5. 手写 DiffPool layer

一个 layer 有两条独立 GCN 支路：embedding GNN 产生 $Z$，assignment GNN 产生 $S$。共享参数会无意限制两者表达。`forward` 返回 pooled tensors、mask、两个辅助损失和 assignment，便于审计而不是把关键中间量藏起来。


In [ ]:
class DiffPoolLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_clusters: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if num_clusters <= 0: raise ValueError("num_clusters 必须为正")  # 按当前条件选择后续控制路径。
        self.embed_gcn1 = DenseGCN(input_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.embed_gcn2 = DenseGCN(hidden_dim, output_dim)  # 计算并保存当前步骤的中间状态。
        self.assign_gcn = DenseGCN(input_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.assign_linear = nn.Linear(hidden_dim, num_clusters)  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, adj: torch.Tensor, mask: torch.Tensor):  # 定义本节可复用的核心函数。
        z = self.embed_gcn2(self.embed_gcn1(x, adj, mask), adj, mask)  # 计算并保存当前步骤的中间状态。
        assign_hidden = self.assign_gcn(x, adj, mask)  # 计算并保存当前步骤的中间状态。
        assign_logits = self.assign_linear(assign_hidden)  # 计算并保存当前步骤的中间状态。
        assignment = masked_assignment_softmax51(assign_logits, mask)  # 计算并保存当前步骤的中间状态。
        pooled_x, pooled_adj, pooled_mask, link_loss, entropy_loss = diffpool_tensors51(  # 计算并保存当前步骤的中间状态。
            z, adj, mask, assignment)  # 执行当前语句以推进本节示例。
        return pooled_x, pooled_adj, pooled_mask, link_loss, entropy_loss, assignment  # 返回当前分支计算出的结果。

layer_probe51 = DiffPoolLayer(3, 8, 6, 4)  # 计算并保存当前步骤的中间状态。
lp_x51, lp_a51, lp_m51, lp_link51, lp_ent51, lp_s51 = layer_probe51(  # 计算并保存当前步骤的中间状态。
    train51.x[:2], train51.adj[:2], train51.mask[:2])  # 执行当前语句以推进本节示例。
assert lp_x51.shape == (2, 4, 6) and lp_a51.shape == (2, 4, 4)  # 用受控断言验证关键不变量。
assert torch.allclose(lp_a51, lp_a51.transpose(1, 2), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(lp_s51.sum(-1)[train51.mask[:2]], torch.ones(int(train51.mask[:2].sum())), atol=1e-6)  # 用受控断言验证关键不变量。
assert float(lp_link51) >= 0 and float(lp_ent51) >= 0  # 用受控断言验证关键不变量。
assert layer_probe51.embed_gcn1.linear.weight.data_ptr() != layer_probe51.assign_gcn.linear.weight.data_ptr()  # 用受控断言验证关键不变量。


## 6. 两层 hierarchical pooling 与图级读出

第一层把最多 8 个节点软聚合为 4 个 cluster，第二层再聚合为 2 个 super-cluster。最终对有效 super-cluster 做 masked mean，得到 `[B,D]`，分类头输出 `[B,2]`。

pool 后邻接通常是稠密加权图，第二层不能再用“原始邻接必须二值/零对角”的校验规则；但对称性、非负性和 padding 清零仍是硬合同。


In [ ]:
class HierarchicalDiffPool(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim: int = 3, hidden_dim: int = 16,  # 定义本节可复用的核心函数。
                 embed_dim: int = 12, clusters1: int = 4, clusters2: int = 2,  # 计算并保存当前步骤的中间状态。
                 num_classes: int = 2):  # 计算并保存当前步骤的中间状态。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.pool1 = DiffPoolLayer(input_dim, hidden_dim, embed_dim, clusters1)  # 计算并保存当前步骤的中间状态。
        self.pool2 = DiffPoolLayer(embed_dim, hidden_dim, embed_dim, clusters2)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(embed_dim, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, batch: DenseBatch, return_debug: bool = False):  # 定义本节可复用的核心函数。
        x1, a1, m1, link1, ent1, s1 = self.pool1(batch.x, batch.adj, batch.mask)  # 计算并保存当前步骤的中间状态。
        x2, a2, m2, link2, ent2, s2 = self.pool2(x1, a1, m1)  # 计算并保存当前步骤的中间状态。
        graph_repr = (x2 * m2[:, :, None]).sum(1) / m2.sum(1, keepdim=True).clamp_min(1)  # 计算并保存当前步骤的中间状态。
        logits = self.classifier(graph_repr)  # 计算并保存当前步骤的中间状态。
        aux = {"link": link1 + link2, "entropy": ent1 + ent2}  # 计算并保存当前步骤的中间状态。
        if (not torch.isfinite(graph_repr).all() or not torch.isfinite(logits).all()  # 按当前条件选择后续控制路径。
                or not torch.isfinite(aux["link"]) or not torch.isfinite(aux["entropy"])):  # 执行当前语句以推进本节示例。
            raise ValueError("层次 DiffPool 输出含非有限值")  # 遇到非法合同立即显式失败。
        debug = {"s1": s1, "s2": s2, "mask1": m1, "mask2": m2, "adj2": a2}  # 计算并保存当前步骤的中间状态。
        return (logits, aux, debug) if return_debug else (logits, aux)  # 返回当前分支计算出的结果。

torch.manual_seed(5102)  # 执行当前语句以推进本节示例。
model_probe51 = HierarchicalDiffPool()  # 计算并保存当前步骤的中间状态。
probe_logits51, probe_aux51, probe_debug51 = model_probe51(train51, return_debug=True)  # 计算并保存当前步骤的中间状态。
assert probe_logits51.shape == (24, 2)  # 用受控断言验证关键不变量。
assert probe_debug51["s1"].shape == (24, 8, 4)  # 用受控断言验证关键不变量。
assert probe_debug51["s2"].shape == (24, 4, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(probe_debug51["adj2"], probe_debug51["adj2"].transpose(1, 2), atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.isfinite(probe_aux51["link"] + probe_aux51["entropy"])  # 用受控断言验证关键不变量。


## 7. 节点置换不变性与 padding 不污染

图分类必须对节点编号置换不变：同时置换 $X$ 的节点轴、$A$ 的两个节点轴和 mask，logits 应不变。只扩大 padding 长度也不能改变有效图输出。两项检查能捕捉“只 mask key、不 mask query/归一化/readout”等常见错误。


In [ ]:
model_probe51.eval()  # 执行当前语句以推进本节示例。
single51 = pad_graphs51([split_graphs51["test"][0]], pad_to=8)  # 计算并保存当前步骤的中间状态。
n51 = int(single51.mask[0].sum())  # 计算并保存当前步骤的中间状态。
valid_perm51 = torch.arange(n51 - 1, -1, -1)  # 计算并保存当前步骤的中间状态。
perm51 = torch.cat([valid_perm51, torch.arange(n51, single51.x.shape[1])])  # 计算并保存当前步骤的中间状态。
permuted51 = DenseBatch(  # 计算并保存当前步骤的中间状态。
    single51.x[:, perm51], single51.adj[:, perm51][:, :, perm51], single51.mask[:, perm51],  # 执行当前语句以推进本节示例。
    single51.labels, single51.graph_ids, single51.snapshot_digest,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    original_logit51 = model_probe51(single51)[0]  # 计算并保存当前步骤的中间状态。
    permuted_logit51 = model_probe51(permuted51)[0]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(original_logit51, permuted_logit51, atol=1e-5)  # 用受控断言验证关键不变量。

pair_pad8_51 = pad_graphs51(split_graphs51["test"][:2], pad_to=8)  # 计算并保存当前步骤的中间状态。
pair_pad11_51 = pad_graphs51(split_graphs51["test"][:2], pad_to=11)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    logits_pad8_51 = model_probe51(pair_pad8_51)[0]  # 计算并保存当前步骤的中间状态。
    logits_pad11_51 = model_probe51(pair_pad11_51)[0]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(logits_pad8_51, logits_pad11_51, atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(pair_pad11_51.x[:, 8:]) == 0  # 用受控断言验证关键不变量。
assert pair_pad8_51.snapshot_digest != pair_pad11_51.snapshot_digest  # 用受控断言验证关键不变量。


## 8. 受控训练与模型选择

目标为 `cross_entropy + 0.03*link_loss + 0.002*entropy`。训练仅访问 train batch；每 5 步在 validation 上选择 checkpoint；加载最佳参数后才评 test。辅助项权重是 recipe 的一部分，变更后应重新发布而非静默沿用旧阈值。


In [ ]:
def accuracy51(model, batch):  # 定义本节可复用的核心函数。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad(): pred = model(batch)[0].argmax(-1)  # 在受管理的上下文中执行操作。
    return float((pred == batch.labels).float().mean())  # 返回当前分支计算出的结果。

torch.manual_seed(5103)  # 执行当前语句以推进本节示例。
model51 = HierarchicalDiffPool()  # 计算并保存当前步骤的中间状态。
optimizer51 = torch.optim.Adam(model51.parameters(), lr=0.025)  # 计算并保存当前步骤的中间状态。
best_val51, best_state51, best_step51, ce_trace51 = -1.0, None, None, []  # 计算并保存当前步骤的中间状态。
for step in range(101):  # 遍历输入元素以累积或检查结果。
    model51.train(); optimizer51.zero_grad()  # 执行当前语句以推进本节示例。
    logits, aux = model51(train51)  # 计算并保存当前步骤的中间状态。
    ce = F.cross_entropy(logits, train51.labels)  # 计算并保存当前步骤的中间状态。
    loss = ce + 0.03 * aux["link"] + 0.002 * aux["entropy"]  # 计算并保存当前步骤的中间状态。
    loss.backward(); torch.nn.utils.clip_grad_norm_(model51.parameters(), 5.0); optimizer51.step()  # 执行当前语句以推进本节示例。
    ce_trace51.append(float(ce.detach()))  # 执行当前语句以推进本节示例。
    if step % 5 == 0:  # 按当前条件选择后续控制路径。
        val_score = accuracy51(model51, val51)  # 计算并保存当前步骤的中间状态。
        if val_score > best_val51:  # 按当前条件选择后续控制路径。
            best_val51 = val_score; best_state51 = copy.deepcopy(model51.state_dict()); best_step51 = step  # 计算并保存当前步骤的中间状态。
model51.load_state_dict(best_state51)  # 执行当前语句以推进本节示例。

assert len(ce_trace51) == 101  # 用受控断言验证关键不变量。
assert min(ce_trace51) < ce_trace51[0] * 0.40  # 用受控断言验证关键不变量。
assert ce_trace51[-1] < ce_trace51[0] * 0.75  # 用受控断言验证关键不变量。
assert best_val51 == 1.0  # 用受控断言验证关键不变量。
assert best_step51 is not None and best_step51 % 5 == 0  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model51.parameters())  # 用受控断言验证关键不变量。
assert all(torch.isfinite(p).all() for p in model51.parameters())  # 用受控断言验证关键不变量。


## 9. 冻结测试与中间合同复核

最终同时检查 train/validation/test accuracy，以及 test assignment 行和和 pooled adjacency 对称性。受控图由固定规则生成，因此 100% 只说明实现学会该规则；真实任务仍需跨来源、跨时间外推与置信区间。


In [ ]:
train_acc51 = accuracy51(model51, train51)  # 计算并保存当前步骤的中间状态。
val_acc51 = accuracy51(model51, val51)  # 计算并保存当前步骤的中间状态。
test_acc51 = accuracy51(model51, test51)  # 计算并保存当前步骤的中间状态。
model51.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad(): final_logits51, final_aux51, final_debug51 = model51(test51, return_debug=True)  # 在受管理的上下文中执行操作。
assert train_acc51 == 1.0 and val_acc51 == 1.0 and test_acc51 == 1.0  # 用受控断言验证关键不变量。
assert final_logits51.shape == (6, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(final_debug51["s1"].sum(-1)[test51.mask], torch.ones(int(test51.mask.sum())), atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.allclose(final_debug51["adj2"], final_debug51["adj2"].transpose(1, 2), atol=1e-5)  # 用受控断言验证关键不变量。
assert float(final_aux51["link"]) >= 0 and float(final_aux51["entropy"]) >= 0  # 用受控断言验证关键不变量。


## 10. 制品合同与包外 publisher registry

manifest 绑定两层 cluster 数、图/schema 规则、train/validation/test snapshot、graph ID split、padding/归一化算法和完整训练 recipe。canonical state digest 逐项覆盖参数 key、dtype、shape 与 bytes。loader 返回 `PublishedDiffPool` evaluator；它根据完整 graph ID 序列确定登记 split，并重算 x/adj/mask/labels/graph_ids 摘要，不能靠调用方填写的 `snapshot_digest` 自证。

攻击者能替换整个 package 并重算内部 hash，故内部 hash 只能检测传输损坏。loader 还检查 package 外只读 publisher registry；伪造包无法自行更新该信任锚。这个发布 evaluator 只复核登记的离线快照；在线新图分类应使用另一个明确版本化的 schema/feature 服务 API，而不是携带伪 snapshot 字段。


In [ ]:
RELEASE51 = "diffpool-demo-51/v1"  # 计算并保存当前步骤的中间状态。
CONFIG51 = {"input_dim": 3, "hidden_dim": 16, "embed_dim": 12,  # 计算并保存当前步骤的中间状态。
            "clusters1": 4, "clusters2": 2, "num_classes": 2}  # 执行当前语句以推进本节示例。
MANIFEST51 = {  # 计算并保存当前步骤的中间状态。
    "config": CONFIG51,  # 执行当前语句以推进本节示例。
    "snapshots": {"train": train51.snapshot_digest, "val": val51.snapshot_digest, "test": test51.snapshot_digest},  # 执行当前语句以推进本节示例。
    "split": {s: [g.graph_id for g in split_graphs51[s]] for s in ("train", "val", "test")},  # 执行当前语句以推进本节示例。
    "runtime": {"policy": "registered_snapshot_evaluator_only",  # 执行当前语句以推进本节示例。
                "digest_fields": ["x", "adj", "mask", "labels", "graph_ids"]},  # 执行当前语句以推进本节示例。
    "schema": {"feature_dim": 3, "undirected": True, "raw_diagonal": 0,  # 执行当前语句以推进本节示例。
               "cross_graph_edges": "reject", "symmetry_atol": SYMMETRY_ATOL51,  # 执行当前语句以推进本节示例。
               "symmetry_rtol": 0.0, "max_adj_weight": MAX_ADJ_WEIGHT51,  # 执行当前语句以推进本节示例。
               "link_stabilization": "per_graph_max_abs_error_scale"},  # 执行当前语句以推进本节示例。
    "recipe": {"seed": SEED, "steps": 101, "optimizer": "Adam", "lr": 0.025,  # 执行当前语句以推进本节示例。
               "loss": "CE+0.03*link+0.002*entropy", "normalization": "D^-1/2(A+I_valid)D^-1/2",  # 执行当前语句以推进本节示例。
               "grad_clip_norm": 5.0, "validation_interval": 5,  # 执行当前语句以推进本节示例。
               "selection": "best_validation_accuracy", "tie_break": "first_strict_improvement",  # 执行当前语句以推进本节示例。
               "checkpoint_step": best_step51},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
state51 = {k: v.detach().cpu().clone() for k, v in model51.state_dict().items()}  # 计算并保存当前步骤的中间状态。
package51 = {"release_id": RELEASE51, "manifest": copy.deepcopy(MANIFEST51), "state": state51}  # 计算并保存当前步骤的中间状态。
package51["state_digest"] = state_digest51(state51)  # 计算并保存当前步骤的中间状态。
package51["package_digest"] = canonical_digest({"release_id": RELEASE51, "manifest": package51["manifest"],  # 计算并保存当前步骤的中间状态。
                                                  "state_digest": package51["state_digest"]})  # 执行当前语句以推进本节示例。
_PUBLISHER_REGISTRY51 = MappingProxyType({RELEASE51: package51["package_digest"]})  # 计算并保存当前步骤的中间状态。

def validate_registered_batch51(batch: DenseBatch, manifest: dict) -> str:  # 定义本节可复用的核心函数。
    if not isinstance(batch, DenseBatch): raise ValueError("输入必须是 DenseBatch")  # 按当前条件选择后续控制路径。
    validate_dense_tensors51(batch.x, batch.adj, batch.mask)  # 执行当前语句以推进本节示例。
    b = batch.x.shape[0]  # 计算并保存当前步骤的中间状态。
    if batch.labels.shape != (b,) or batch.labels.dtype != torch.long or batch.labels.device != batch.x.device:  # 按当前条件选择后续控制路径。
        raise ValueError("DenseBatch labels 合同不匹配")  # 遇到非法合同立即显式失败。
    if (len(batch.graph_ids) != b or len(set(batch.graph_ids)) != b  # 按当前条件选择后续控制路径。
            or any(not isinstance(g, str) or not g for g in batch.graph_ids)):  # 执行当前语句以推进本节示例。
        raise ValueError("DenseBatch graph_ids 合同不匹配")  # 遇到非法合同立即显式失败。
    matching = [split for split, ids in manifest["split"].items() if tuple(ids) == tuple(batch.graph_ids)]  # 计算并保存当前步骤的中间状态。
    if len(matching) != 1: raise ValueError("graph_ids 未唯一匹配发布 split")  # 按当前条件选择后续控制路径。
    split = matching[0]  # 计算并保存当前步骤的中间状态。
    actual = dense_snapshot_digest51(batch.x, batch.adj, batch.mask, batch.labels, batch.graph_ids)  # 计算并保存当前步骤的中间状态。
    if not isinstance(batch.snapshot_digest, str) or batch.snapshot_digest != actual:  # 按当前条件选择后续控制路径。
        raise ValueError("snapshot 自带摘要与 tensor/metadata 不一致")  # 遇到非法合同立即显式失败。
    if actual != manifest["snapshots"][split]:  # 按当前条件选择后续控制路径。
        raise ValueError("snapshot 未在发布 manifest 登记")  # 遇到非法合同立即显式失败。
    return split  # 返回当前分支计算出的结果。

class PublishedDiffPool(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, model: HierarchicalDiffPool, manifest: dict):  # 定义本节可复用的核心函数。
        super().__init__(); self.model = model; self._manifest = copy.deepcopy(manifest)  # 计算并保存当前步骤的中间状态。

    def forward(self, batch: DenseBatch, return_debug: bool = False):  # 定义本节可复用的核心函数。
        validate_registered_batch51(batch, self._manifest)  # 执行当前语句以推进本节示例。
        return self.model(batch, return_debug=return_debug)  # 返回当前分支计算出的结果。

def load_published_diffpool(package: dict) -> PublishedDiffPool:  # 定义本节可复用的核心函数。
    if set(package) != {"release_id", "manifest", "state", "state_digest", "package_digest"}:  # 按当前条件选择后续控制路径。
        raise ValueError("package 字段集合非法")  # 遇到非法合同立即显式失败。
    release_id = package["release_id"]  # 计算并保存当前步骤的中间状态。
    if release_id not in _PUBLISHER_REGISTRY51: raise ValueError("未知 release")  # 按当前条件选择后续控制路径。
    actual_state = state_digest51(package["state"])  # 计算并保存当前步骤的中间状态。
    actual_package = canonical_digest({"release_id": release_id, "manifest": package["manifest"],  # 计算并保存当前步骤的中间状态。
                                       "state_digest": actual_state})  # 执行当前语句以推进本节示例。
    if actual_state != package["state_digest"] or actual_package != package["package_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("包内摘要不匹配")  # 遇到非法合同立即显式失败。
    if actual_package != _PUBLISHER_REGISTRY51[release_id]:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry 信任锚不匹配")  # 遇到非法合同立即显式失败。
    if package["manifest"] != MANIFEST51:  # 按当前条件选择后续控制路径。
        raise ValueError("config/snapshot/split/schema/recipe 合同不匹配")  # 遇到非法合同立即显式失败。
    model = HierarchicalDiffPool(**package["manifest"]["config"])  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(package["state"], strict=True); model.eval()  # 计算并保存当前步骤的中间状态。
    restored = PublishedDiffPool(model, package["manifest"]); restored.eval()  # 计算并保存当前步骤的中间状态。
    return restored  # 返回当前分支计算出的结果。

restored51 = load_published_diffpool(package51)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(restored51(test51)[0], model51(test51)[0], atol=1e-7)  # 用受控断言验证关键不变量。
assert isinstance(restored51, PublishedDiffPool)  # 用受控断言验证关键不变量。
assert package51["manifest"]["snapshots"]["train"] == train51.snapshot_digest  # 用受控断言验证关键不变量。

fake_digest51 = DenseBatch(test51.x, test51.adj, test51.mask, test51.labels, test51.graph_ids, "attacker-digest")  # 计算并保存当前步骤的中间状态。
fake_labels_draft51 = DenseBatch(test51.x, test51.adj, test51.mask, 1 - test51.labels, test51.graph_ids, "pending")  # 计算并保存当前步骤的中间状态。
fake_labels51 = DenseBatch(fake_labels_draft51.x, fake_labels_draft51.adj, fake_labels_draft51.mask,  # 计算并保存当前步骤的中间状态。
                           fake_labels_draft51.labels, fake_labels_draft51.graph_ids,  # 执行当前语句以推进本节示例。
                           dense_snapshot_digest51(fake_labels_draft51.x, fake_labels_draft51.adj,  # 执行当前语句以推进本节示例。
                                                   fake_labels_draft51.mask, fake_labels_draft51.labels,  # 执行当前语句以推进本节示例。
                                                   fake_labels_draft51.graph_ids))  # 执行当前语句以推进本节示例。
snapshot_rejections51 = {}  # 计算并保存当前步骤的中间状态。
for attack_name, attacked in {"digest": fake_digest51, "labels": fake_labels51}.items():  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        restored51(attacked)  # 执行当前语句以推进本节示例。
        raise AssertionError(f"伪造 snapshot {attack_name} 被接受")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        snapshot_rejections51[attack_name] = str(exc)  # 计算并保存当前步骤的中间状态。
assert set(snapshot_rejections51) == {"digest", "labels"}  # 用受控断言验证关键不变量。

forged51 = copy.deepcopy(package51)  # 计算并保存当前步骤的中间状态。
forged51["manifest"]["recipe"]["loss"] = "attacker-controlled"  # 计算并保存当前步骤的中间状态。
first_key51 = next(iter(forged51["state"]))  # 计算并保存当前步骤的中间状态。
forged51["state"][first_key51] = forged51["state"][first_key51] + 0.02  # 计算并保存当前步骤的中间状态。
forged51["state_digest"] = state_digest51(forged51["state"])  # 计算并保存当前步骤的中间状态。
forged51["package_digest"] = canonical_digest({"release_id": RELEASE51, "manifest": forged51["manifest"],  # 计算并保存当前步骤的中间状态。
                                                 "state_digest": forged51["state_digest"]})  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_published_diffpool(forged51)  # 执行当前语句以推进本节示例。
    raise AssertionError("整体替换 manifest 并重算内部摘要后仍被接受")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(exc)  # 用受控断言验证关键不变量。


## 11. 复杂度、失败模式与生产差距

- **复杂度**：dense GCN 为 $O(BN^2D)$；一次 pooling 的 $S^TAS$ 约 $O(BN^2K+BNK^2)$。大图必须稀疏化、分区或采样，不能直接 padding 到全局最大节点数。
- **失败模式**：padding 行参与 softmax；只置换 feature 没同步 adjacency；非对称邻接静默进入；空 cluster 除零；跨图边污染；把 test 图参与 assignment 预训练。
- **模型风险**：cluster 缺少可解释语义，entropy/link 权重不当会导致全部节点挤进一个 cluster 或均匀分配。应监控 cluster mass、熵、稳定性和 OOD 图规模。
- **发布差距**：真实系统还需特征版本、图快照水位、稀疏算子 ABI、数值容差、签名/KMS、灰度和回滚。


In [ ]:
assert test_acc51 == 1.0  # 用受控断言验证关键不变量。
assert package51["package_digest"] == _PUBLISHER_REGISTRY51[RELEASE51]  # 用受控断言验证关键不变量。
assert set(MANIFEST51["snapshots"]) == {"train", "val", "test"}  # 用受控断言验证关键不变量。
assert final_debug51["mask2"].shape == (6, 2)  # 用受控断言验证关键不变量。
assert all(parameter.device.type == "cpu" for parameter in restored51.parameters())  # 用受控断言验证关键不变量。
